# 39. Open RAG Eval - Reference-free Evaluation

Evaluate without golden answers: grade retrieved passages (UMBRELA-style) and check answer nuggets.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

## Why this matters

Golden answers are expensive. **Open RAG Eval** (Vectara) evaluates *without* ground-truth answers by combining: **UMBRELA** (grade each retrieved passage 0-3), **AutoNuggetizer** (list the facts a good answer must contain and check coverage), citation checks and hallucination detection. Here we implement the core ideas locally so you can see each signal.

## How it works

1. UMBRELA: grade every retrieved passage 0-3 for relevance
2. Nuggetize: LLM lists atomic facts the answer should contain
3. Check which nuggets the generated answer covers (nugget recall)
4. Split the answer into claims and verify each against the context (hallucination check)
5. Aggregate across queries

```text
 query --> retrieve --> UMBRELA (0-3 per passage)  => retrieval quality, no gold answer needed
       --> nuggets (needed facts) --> covered by answer? => nugget recall
       --> answer sentences --> supported by context?    => hallucination rate
```

## Prerequisites

Needs a running Ollama server (`ollama serve`) with `llama3.2` pulled, plus the shared helpers in `../common.py`.

**Extra requirements for this notebook:** Optional: `pip install open-rag-eval` for the full Vectara framework

## Setup

Load the handbook, split it into ~500-character chunks (50 overlap) and index them. Every notebook starts from this same baseline so results are comparable.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

[open-rag-eval](https://github.com/vectara/open-rag-eval) evaluates RAG **without ground-truth answers** by combining: retrieval relevance (UMBRELA), nugget coverage (AutoNuggetizer), citation checks and hallucination detection. This notebook re-implements the core ideas locally so you can see how they work.

## 1. UMBRELA-style retrieval grading
Each retrieved passage gets 0-3: 0 unrelated, 1 related but no answer, 2 partial answer, 3 dedicated answer.

In [ ]:
def umbrela(query, passage):
    return llm_score(
        "Grade the passage for the query: 0 = unrelated, 1 = related but does not answer, "
        "2 = partially answers, 3 = dedicated, exact answer.\n\n"
        f"Query: {query}\n\nPassage: {passage}", lo=0, hi=3) or 0

query = "How does hybrid search merge keyword and vector results?"
hits = store.search(query, 5)
grades = [umbrela(query, h["text"]) for h in hits]
print("grades:", grades, "-> mean", sum(grades) / len(grades))

## 2. Nugget evaluation
Break the *information need* into atomic nuggets (facts a good answer must contain), then check which ones the answer covers.

In [ ]:
nuggets = llm_json(f"List 4 atomic facts a complete answer to this question must contain. JSON list of strings.\n\nQuestion: {query}") or []
response = answer(query, [h["text"] for h in hits])

covered = {n: llm_yes(f"Answer: {response}\n\nDoes the answer clearly state this fact: {n}") for n in nuggets}
for n, ok in covered.items(): print("[x]" if ok else "[ ]", n)
print(f"nugget recall: {sum(covered.values()) / max(len(covered), 1):.2f}")

## 3. Hallucination / citation check
Split the answer into sentences and verify each one against the retrieved context.

In [ ]:
ctx = "\n".join(h["text"] for h in hits)
sentences = split_sentences(response)
supported = [llm_yes(f"Context:\n{ctx}\n\nClaim: {s}\nIs the claim supported by the context?") for s in sentences]
print(f"{sum(supported)}/{len(supported)} sentences supported by retrieved context")

## Run the reference-free metrics over several queries

In [ ]:
import pandas as pd
rows = []
for item in load_eval_questions()[:5]:
    q = item["question"]
    hits = store.search(q, 4)
    ctx = "\n".join(h["text"] for h in hits)
    resp = answer(q, [h["text"] for h in hits])
    grades = [umbrela(q, h["text"]) for h in hits]
    nug = llm_json(f"List 3 atomic facts a complete answer to this question must contain. JSON list of strings.\n\nQuestion: {q}") or []
    cov = [llm_yes(f"Answer: {resp}\n\nDoes the answer state this fact: {n}") for n in nug if isinstance(n, str)]
    sup = [llm_yes(f"Context:\n{ctx}\n\nClaim: {s}\nIs the claim supported?") for s in split_sentences(resp)]
    rows.append({"question": q[:45], "mean_umbrela": round(sum(grades) / len(grades), 2),
                 "nugget_recall": round(sum(cov) / max(len(cov), 1), 2), "supported_claims": round(sum(sup) / max(len(sup), 1), 2)})
rep = pd.DataFrame(rows).set_index("question"); display(rep); print(rep.mean().round(2))

## Do the signals agree with the golden-set metric?
Reference-free scores are only useful if they correlate with real quality. Compare them with keyword recall against the known answers.

In [ ]:
gold = [keyword_recall(i["answer"], answer(i["question"], [h["text"] for h in store.search(i["question"], 4)])) for i in load_eval_questions()[:5]]
rep["gold_keyword_recall"] = gold
display(rep[["nugget_recall", "supported_claims", "gold_keyword_recall"]])
print(rep[["mean_umbrela", "nugget_recall", "supported_claims", "gold_keyword_recall"]].corr().round(2)["gold_keyword_recall"])

## Using the real framework
For production, install the full framework (`pip install open-rag-eval`) which adds connectors for Vectara, LlamaIndex and LangChain, report generation and a UI. See https://github.com/vectara/open-rag-eval.

## Trade-offs

| Strengths | Weaknesses |
|---|---|
| No golden answers needed | Judge-model noise |
| Multiple complementary signals | Nuggets can miss facts |
| Works on production traffic | Correlation with human judgement must still be validated |

## When to use it

Evaluating on live or unlabelled queries, or comparing configurations when labelling is too costly.

## Experiments to try

- Validate the signals against human labels on 20 queries.
- Add a citation-accuracy check.
- Track the metrics per release.

## Key takeaways

- Grade retrieval, nuggets and claims separately.
- Reference-free metrics need calibration.
- Use them to monitor production.